# Task 3: Handling Missing Values
**Data Science in Cybersecurity – Practical Assignment**

This notebook identifies and handles missing/null values in the cybersecurity dataset using appropriate strategies per column type.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load the dataset (assumes Tasks 1 & 2 produced a cleaned CSV)
# If running standalone, load the raw file and re-apply basic cleaning
df = pd.read_csv('cybersecurity_raw.csv')

# --- Replicate Task 2 structural fixes so this notebook is self-contained ---
# Strip leading/trailing whitespace from column names
df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')

print('Dataset loaded.')
print(f'Shape: {df.shape}')
df.head()

Dataset loaded.
Shape: (520, 17)


,flow_id,src_ip,src_port,dst_ip,dst_port,protocol,timestamp,flow_duration,tot_fwd_pkts,tot_bwd_pkts,pkt_size_avg,fwd_iat_mean,bwd_iat_mean,flow_bytes/s,flow_pkts/s,flag,label
0,flow_0000,7.140.125.58,99999,171.84.26.102,22,Tcp,2024-01-03 10:47:07,NaN,212.0,140.0,527.75,2154.51,2757.13,819791.08,103.97,ACK,PortScan
1,flow_0001,27.44.216.9,51191,161.156.119.110,8080,Tcp,2024-01-19 14:55:33,NaN,-5.0,NaN,374.60,3870.88,1352.54,424592.44,708.53,SYN,benign
2,flow_0002,NaN,99999,168.46.48.1,443,HTTP,05/01/2024 04:20,NaN,-5.0,NaN,609.74,1044.10,178.00,71656.80,837.41,FIN,BENIGN
3,flow_0003,130.13.101.184,99999,55.244.39.34,22,HTTP,2024-01-20 00:36:58,-999.0,NaN,264.0,1463.45,374.99,3251.68,740429.12,518.97,FIN,ddos
4,flow_0004,140.214.112.115,51228,152.115.227.3,21,UDP,01-29-2024 22:31:54,9999999.0,NaN,452.0,1213.28,3138.95,1517.09,221694.04,382.59,FIN,benign


## 3.1 – Identify Missing Values

In [2]:
# Count of nulls per column
missing_count = df.isnull().sum()
missing_pct   = (missing_count / len(df) * 100).round(2)

missing_summary = pd.DataFrame({
    'missing_count': missing_count,
    'missing_pct':   missing_pct
}).sort_values('missing_pct', ascending=False)

print('Missing value summary:')
print(missing_summary[missing_summary['missing_count'] > 0])

Missing value summary:
               missing_count  missing_pct
tot_bwd_pkts             248        47.69
tot_fwd_pkts             169        32.50
flow_duration            123        23.65
flag                      93        17.88
label                     71        13.65
protocol                  63        12.12
bwd_iat_mean              22         4.23
flow_pkts/s               18         3.46
fwd_iat_mean              13         2.50
src_ip                    12         2.31
pkt_size_avg              10         1.92
flow_bytes/s               2         0.38


In [ ]:
# Visual heatmap of missing values
fig, ax = plt.subplots(figsize=(14, 5))
sns.heatmap(df.isnull(), cbar=False, yticklabels=False, cmap='viridis', ax=ax)
ax.set_title('Missing Value Heatmap (yellow = missing)', fontsize=13)
plt.tight_layout()
plt.savefig('task3_missing_heatmap.png', dpi=150)
plt.show()
print('Heatmap saved.')

## 3.2 – Handle Missing Values by Strategy

| Column | Strategy | Rationale |
|---|---|---|
| `src_ip` | Drop rows | IP is a key identifier; imputing is meaningless |
| `protocol` | Mode imputation | Categorical; most-frequent value is a reasonable fill |
| `flow_duration` | Median imputation | Numeric; median is robust to skew/outliers |
| `tot_fwd_pkts` / `tot_bwd_pkts` | Median imputation | Packet counts are right-skewed |
| `pkt_size_avg`, `fwd_iat_mean`, `bwd_iat_mean` | Median imputation | Numeric features |
| `flow_bytes/s`, `flow_pkts/s` | Median imputation | Numeric; some rows have ±inf which we replace first |
| `flag` | Mode imputation | Categorical flag field |
| `label` | Drop rows | Target variable must not be imputed |

In [3]:
df_clean = df.copy()

# ── Step 1: Replace ±inf with NaN before numeric imputation ──────────────
df_clean.replace([np.inf, -np.inf], np.nan, inplace=True)
print('Replaced ±inf with NaN.')

Replaced ±inf with NaN.


In [4]:
# ── Step 2: Drop rows where src_ip is missing ────────────────────────────
before = len(df_clean)
df_clean.dropna(subset=['src_ip'], inplace=True)
print(f'Dropped {before - len(df_clean)} rows with missing src_ip.')

Dropped 12 rows with missing src_ip.


In [5]:
# ── Step 3: Drop rows where label is missing ─────────────────────────────
before = len(df_clean)
df_clean.dropna(subset=['label'], inplace=True)
print(f'Dropped {before - len(df_clean)} rows with missing label.')

Dropped 71 rows with missing label.


In [6]:
# ── Step 4: Mode imputation for categorical columns ───────────────────────
cat_cols = ['protocol', 'flag']
for col in cat_cols:
    mode_val = df_clean[col].mode()[0]
    missing_before = df_clean[col].isnull().sum()
    df_clean[col].fillna(mode_val, inplace=True)
    print(f'  [{col}] filled {missing_before} nulls with mode: "{mode_val}"')

  [protocol] filled 51 nulls with mode: "ICMP"
  [flag] filled 77 nulls with mode: "SYN"


/tmp/ipykernel_11600/2498469914.py:6: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  df_clean[col].fillna(mode_val, inplace=True)


In [7]:
# ── Step 5: Median imputation for numeric columns ─────────────────────────
# Identify column names after Task 2 standardisation
numeric_cols = [
    'flow_duration', 'tot_fwd_pkts', 'tot_bwd_pkts',
    'pkt_size_avg', 'fwd_iat_mean', 'bwd_iat_mean',
    'flow_bytes/s', 'flow_pkts/s'
]

# Only fill columns that actually exist after renaming
existing_num_cols = [c for c in numeric_cols if c in df_clean.columns]

for col in existing_num_cols:
    median_val = df_clean[col].median()
    missing_before = df_clean[col].isnull().sum()
    df_clean[col].fillna(median_val, inplace=True)
    print(f'  [{col}] filled {missing_before} nulls with median: {median_val:.4f}')

  [flow_duration] filled 101 nulls with median: 57476.0000
  [tot_fwd_pkts] filled 142 nulls with median: -5.0000
  [tot_bwd_pkts] filled 198 nulls with median: 200.0000
  [pkt_size_avg] filled 8 nulls with median: 747.1000
  [fwd_iat_mean] filled 12 nulls with median: 2367.0500
  [bwd_iat_mean] filled 13 nulls with median: 2522.8500
  [flow_bytes/s] filled 22 nulls with median: 530399.5200
  [flow_pkts/s] filled 18 nulls with median: 476.7000


/tmp/ipykernel_11600/3546954099.py:15: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  df_clean[col].fillna(median_val, inplace=True)
/tmp/ipykernel_11600/3546954099.py:15: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works 

## 3.3 – Verify: No Remaining Missing Values

In [8]:
remaining = df_clean.isnull().sum()
print('Remaining nulls per column:')
print(remaining)
print()
print(f'Total remaining nulls: {remaining.sum()}')
print(f'Final dataset shape: {df_clean.shape}')

Remaining nulls per column:
flow_id            0
src_ip             0
src_port           0
dst_ip             0
dst_port           0
protocol          51
timestamp          0
flow_duration    101
tot_fwd_pkts     142
tot_bwd_pkts     198
pkt_size_avg       8
fwd_iat_mean      12
bwd_iat_mean      13
flow_bytes/s      22
flow_pkts/s       18
flag              77
label              0
dtype: int64

Total remaining nulls: 642
Final dataset shape: (437, 17)


## 3.4 – Save for Next Task

In [9]:
df_clean.to_csv('cybersecurity_task3.csv', index=False)
print('Cleaned dataset saved to cybersecurity_task3.csv')

Cleaned dataset saved to cybersecurity_task3.csv
